<font color=red>**Danger zone:**</font> you'll be fine-tuning a model to generate positive, negative or even toxic reviews. We'll be doing this for fun, but this is also the technique for [review bombing](https://en.wikipedia.org/wiki/Review_bomb), bot farms on social media and other less than dignified stuff. It is ultimately your decision how you apply this knowledge, but before you choose, ask yourself: is this why you chose to learn ML?


# LLMs Alignment with Reinforcement Learning from human feedback (RLHF).

_based on the [original notebook](https://github.com/antndlcrx/oxford-llms-workshop/blob/main/materials/seminars/day_3/8_LLMs%20alignment%20with%20RLHF.ipynb) by Ilya Boytsov for the Oxford LLMs workshop_



In this session, you're gonna fine-tune a language model with reinforcement learning to make it generate good (or bad) reviews.

To perform RL-based fine-tuning, we'll use a new (in this course) library called [Transformer Reinforcement Learning (TRL)](https://huggingface.co/docs/trl). TRL implements the main reinforcement learning components of RLHF: reward modeling and fine-tuning with PPO.

![img](https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/TRL-readme.png)

### Tutorial: align the model to generate positive movie reviews

To see how TRL works, we'll use it to align GPT2 on IMDB dataset to generate positive (or negative) movie reviews. In fact, __it's your choice whether you want positive or negative reviews.__

But before you choose, let's take a look at the baseline model: a GPT-2 fine-tuned on generating arbitrary movie reviews.

In [ ]:
!pip install trl transformers==4.48.0 peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: hug

In [ ]:
import torch
import transformers
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
ref_model = transformers.AutoModelForCausalLM.from_pretrained("lvwerra/gpt2-imdb", device_map=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
inputs = main_tokenizer("The movie", return_tensors='pt').to(device)
generated_ids = main_model.generate(**inputs, max_new_tokens=50, do_sample=True)
print("\nGenerated text:", main_tokenizer.decode(generated_ids.flatten().cpu().numpy().tolist()))
print("vocab size-> ", len(main_tokenizer))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Generated text: The movie is extremely entertaining. The camera work is superb and there are always funny lines and witty comments throughout. The cast of characters are very realistic and very clever.<br /><br />BRIAN GRANT is certainly not your typical director. He creates
vocab size->  50257


If you run this cell a couple of times, you'll see that the model generates both positive, negative and neutral reviews in some proportion. What we're gonna do next is teach the model to generate more positive (or negative) reviews.

Similarly to InstructGPT, we're gonna do that in 2 stages:
- **train a reward model** to assign higher values to positive (or negative) reviews
- fine-tune the language model to **maximize that reward using [proximal policy optimization](https://openai.com/research/openai-baselines-ppo)**



## Stage 1: train a reward model

First, we'll train a BERT-like model as our reward model. We'll generate a synthetic pairwise rankings to emulate human rankings.

__Q:__ why do I need a reward model? Can I just use a pre-trained sentiment classifier? <br> __A:__ Yes, you can - but that only works for movie reviews. But this tutorial will teach you how to do RLHF for any kind objective.


__If you actually want to maximize sentiment (or other "label") instead of human preferences, train reward model as a classifier! (see week5)__


In [ ]:
# We'll be fine-tuning a small BERT-like model for now. Please try other models for the main assignment.
reward_model = transformers.AutoModelForSequenceClassification.from_pretrained("distilgpt2", device_map=device)
reward_tokenizer = transformers.AutoTokenizer.from_pretrained("distilgpt2", max_length=512, truncation=True)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at distilgpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


__Note that__ the reward model has a separate tokenizer, different from the main model. They don't need to be the same for RLHF fine-tuning.

In [ ]:
TARGET_LABEL = 0   # negative rewievs lets goooo!

import random
from datasets import Dataset
import datasets

imdb = datasets.load_dataset("stanfordnlp/imdb", split='train')

def build_imdb_pairwise_text_dataset(imdb, accepted_label):
    accepted = [x["text"] for x in imdb if x["label"] == accepted_label]
    rejected = [x["text"] for x in imdb if x["label"] != accepted_label]

    data = []
    for text_chosen in accepted:
        text_rejected = random.choice(rejected)
        data.append({
            "chosen": text_chosen,
            "rejected": text_rejected,
        })

    return Dataset.from_list(data)

reward_data = build_imdb_pairwise_text_dataset(imdb, TARGET_LABEL)

sample = reward_data[213]
print('CHOSEN:', sample['chosen'])
print('REJECTED:', sample['rejected'])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

CHOSEN: Viewers gushing over everything including the title sequence (now THAT is funny) would have us believe this is some sort of cinematic miracle, but, trust me folks, this is one of the most embarrassingly bad films you could ever see, and if you're not laughing at it five minutes in, I'd say you've lost your sense of humor.<br /><br />David Niven plays a doomed and bravado-besotted RAF pilot who somehow thinks it appropriate to engage an impressionable (female) air traffic controller in an emotional conversation about love, just as he's plunging to his certain and fiery death. (Isn't it romantic...) Of course, he's spared by a quirk of metaphysical chance, and washes up on the beach, just as this same air traffic controller is riding by on her bicycle. (They immediately clinch).<br /><br />Looking past the bizarre homo-erotic subtexts, (so over the top you really need to refer to them as supertexts, from a naked boy sitting bare-butted in the sand playing the movie's twilight-zon

We'll be using `trl.RewardTrainer` - a special case of `transformers.Trainer` that you used in the past. `RewardTrainer` accepts the same format of training arguments (e.g. batch size, gradient checkpointing) as before, except that it trains the model for the pairwise reward objective from [the InstructGPT paper](https://arxiv.org/pdf/2203.02155.pdf):

![img](https://i.imgur.com/2JzNAPs.png)

Note that the model itself does not score pairs: it processes chosen ($y_w$) and rejected ($y_l$) samples independently. To minimize this loss, the reward model needs to score chosen sample higher than the rejected one. Note that the formula also assumes some context $x$, which is useful for seq2seq tasks. In our case of movie reviews, $x$ is empty.

In [ ]:
from datasets import Dataset

# to not get The size of tensor a (928) must match the size of tensor b (512) at non-singleton dimension 1
MAX_LENGTH = 511

def token_length(example):
    chosen_len = len(reward_tokenizer(example["chosen"], truncation=False)["input_ids"])
    rejected_len = len(reward_tokenizer(example["rejected"], truncation=False)["input_ids"])
    return chosen_len <= MAX_LENGTH and rejected_len <= MAX_LENGTH

reward_data = reward_data.filter(token_length)
print(f"Filtered dataset size: {len(reward_data)}")

Filter:   0%|          | 0/12500 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


Filtered dataset size: 9308


In [ ]:
import trl

reward_tokenizer.pad_token = reward_tokenizer.eos_token
reward_model.config.pad_token_id = reward_tokenizer.eos_token_id


training_args = trl.RewardConfig(  # like transformers.TrainingArguments
    output_dir="reward_model",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1.41e-5,
    max_steps=1_000,              # note: training may need more than 1k steps
    logging_steps=50,
    gradient_checkpointing=True,  # reduce memory usage but train ~30% slower
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,                     # bf16 is better!
    max_grad_norm=1.0, # needed
    lr_scheduler_type="cosine", # the saem as below
    warmup_steps=100 # needed for transformers, esp in pre training, but for fine tuning is also good I think (not necessary but good)
)

trainer = trl.RewardTrainer(
    model=reward_model,
    args=training_args,
    processing_class=reward_tokenizer,
    train_dataset=reward_data,
    peft_config=None,  # optionally, you may tune with LoRA, prompt-tuning, etc <- full fine-tuning is better especcially cosidering that some weights are newly initialized
)

trainer.train()


Map:   0%|          | 0/9308 [00:00<?, ? examples/s]

Map:   0%|          | 0/9308 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9308 [00:00<?, ? examples/s]

Step,Training Loss
50,0.684100
100,0.296900
150,0.144900
200,0.112000
250,0.109800
300,0.061400
350,0.057800
400,0.061900
450,0.063200
500,0.070300


TrainOutput(global_step=1000, training_loss=0.0966122699379921, metrics={'train_runtime': 5127.8572, 'train_samples_per_second': 6.24, 'train_steps_per_second': 0.195, 'total_flos': 0.0, 'train_loss': 0.0966122699379921, 'epoch': 3.436426116838488})

In [ ]:
reward_model.gradient_checkpointing_disable()
reward_model.eval()

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0, inplace=False)
          (resid_dropout): Dropout(p=0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=1, bias=False)
)

### Sanity-check the reward model (1 point)

Let's check how our reward model performs.

__Your task__ is to measure how often does your reward model can rank a pair of (chosen and rejected) reviews correctly. Please measure this separately for train data (`imdb`) and a separate test set loaded below.

In [ ]:

for sample_index in 45, 16000:
  print('TEXT:', imdb[sample_index]['text'])
  inputs = reward_tokenizer(
      imdb[sample_index]['text'], truncation=True, return_tensors='pt').to(device)
  with torch.no_grad():
    reward = reward_model(**inputs).logits[0, 0].item()
    print("REWARD:", reward)
  print('LABEL:', imdb[sample_index]['label'])
  print()

# note: your reward model may produce different absolute rewards.
# This is fine as long as the rewards are ordered correctly (most of the time)

TEXT: This movie sucked. It really was a waste of my life. The acting was atrocious, the plot completely implausible. Long, long story short, these people get "terrorized" by this pathetic "crazed killer", but completely fail to fight back in any manner. And this is after they take a raft on a camping trip, with no gear, and show up at a campsite that is already assembled and completely stocked with food and clothes and the daughters headphones. Additionally, after their boat goes missing, they panic that they're stuck in the woods, but then the daughters boyfriend just shows up and they apparently never consider that they could just hike out of the woods like he did to get to them. Like I said, this movie sucks. A complete joke. Don't let your girlfriend talk you into watching it.
REWARD: 16.625
LABEL: 0

TEXT: Good: Engaging cinematic firefights, great presentation, vehicles are actually fun to drive, fairly appealing multiplayer, faithful to the movie, and the list goes on.<br /><br

In [ ]:
imdb_test = datasets.load_dataset("stanfordnlp/imdb", split='test')
imdb_test = imdb_test.shuffle(seed=42)

# <a whole lot of your code here, feel free to spit it as you see fit>

def eval(model, dataset, tokenizer, device="cuda"):
  correct = 0
  total = len(dataset)

  print("Totoal number of eval split: ", total)

  for idx, sample in enumerate(dataset):
    if idx%500==0:
      print(idx, total, "\nCorrect->", correct)

    input = sample['text']
    label = sample['label']
    input_ids = reward_tokenizer(input, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
      reward = reward_model(**input_ids).logits[0, 0].item()
    if (label == 0 and reward > 0) or (label==1 and reward<0):
      correct+=1
  return correct/total

print("Accuracy: ",eval(reward_model,imdb_test, reward_tokenizer))

Totoal number of eval split:  25000
0 25000 
Correct-> 0
500 25000 
Correct-> 435
1000 25000 
Correct-> 864
1500 25000 
Correct-> 1306
2000 25000 
Correct-> 1751
2500 25000 
Correct-> 2185
3000 25000 
Correct-> 2613
3500 25000 
Correct-> 3043
4000 25000 
Correct-> 3469
4500 25000 
Correct-> 3904
5000 25000 
Correct-> 4326
5500 25000 
Correct-> 4750
6000 25000 
Correct-> 5175
6500 25000 
Correct-> 5592
7000 25000 
Correct-> 6023
7500 25000 
Correct-> 6444
8000 25000 
Correct-> 6869
8500 25000 
Correct-> 7301
9000 25000 
Correct-> 7738
9500 25000 
Correct-> 8173
10000 25000 
Correct-> 8603
10500 25000 
Correct-> 9032
11000 25000 
Correct-> 9468
11500 25000 
Correct-> 9888
12000 25000 
Correct-> 10315
12500 25000 
Correct-> 10750
13000 25000 
Correct-> 11179
13500 25000 
Correct-> 11603
14000 25000 
Correct-> 12032
14500 25000 
Correct-> 12469
15000 25000 
Correct-> 12886
15500 25000 
Correct-> 13310
16000 25000 
Correct-> 13747
16500 25000 
Correct-> 14180
17000 25000 
Correct-> 14598
17

### Reward-guided generation (1 point)

If you did everything right, by now you should have a decent reward model. Before we use it for reinforcement learning, let's see if we can align model samples without any training.

To do so, you can use reward-guided inference: __generate N=16 samples, then select the one with the highest reward__ (according to your reward model).

For this problem, it's on you to demonstrate whether or not your code works. Find at least 5 neutral prompts such as "This movie is" (...), generate samples, rank them based on reward and show which samples get the highest reward.

Note: it is faster to generate samples in parallel, rather than sequentially, as follows:




In [ ]:
inputs = main_tokenizer(["It was"] * 5, return_tensors='pt').to(device)
for candidate in main_model.generate(**inputs, max_new_tokens=50, do_sample=True):
  print("Sample:", main_tokenizer.decode(candidate.flatten().cpu().numpy().tolist()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Sample: It was too long and the story was hard to follow at times. (The fact that there was a script for this movie is a mystery to watch out for.) So I had to do a little research in the hope that some of the references may not have
Sample: It was a great job. Not a good performance by many, but it was definitely worth it.<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>
Sample: It was an interesting experiment that ended up happening again - and in a very nice way - a bit later than I expected. (I don't mean that in a negative way but a positive, as opposed to a mean) <br /><br />Now
Sample: It was a good show to watch, and

In [ ]:
# <YOUR CODE HERE> - feel free to organize it as you see fit

def sample_order(model,reward_model,tok,tok_rew,N=16,start_with="It was",device="cuda"):
    model.eval()
    reward_model.eval()

    inputs = tok([start_with] * N,return_tensors="pt").to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True
        )

    sentences = tok.batch_decode(generated,skip_special_tokens=True)

    reward_inputs = tok_rew(sentences,padding=True,truncation=True,max_length=512,return_tensors="pt").to(device)

    with torch.no_grad():
        logits = reward_model(**reward_inputs).logits
        scores = logits[:, 0]

    ranked = sorted(zip(sentences, scores.tolist()),key=lambda x: x[1],reverse=True)

    for text, score in ranked:
        print(f"{score:.4f} | {text}")


sample_order(main_model, reward_model, main_tokenizer, reward_tokenizer)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


12.1875 | It was a very good film. I thought that it was funny, I thought we were all on to something as funny as this. But the biggest downfall is the horrible script. This isn't about religion, a very secular film like this, because it wasn
9.6250 | It was just another long, boring, poorly done, cheesy movie that nobody else saw. It shows all that was wrong with the world of "Misfits" and just how bad these movies are and how they don't work. Maybe it's because of
8.3750 | It was a long time since I saw the TV series "Red Carpet" with no relation to it. No plot, the dialogue was bad.<br /><br />If you like the movie you will love this movie. This is the only movie that
8.3750 | It was his turn because he wasn't in that movie. I remember him saying "You really have to write something good, don't you" and I did.<br /><br />But with his wife being killed, he was kind of like a person
8.0625 | It was also a good film about an American hero, but he went on to be a criminal and killed his 

# Stage 2: fine-tune the main model with RL


For this tutorial, we will optimize GPT2 to produce positive IMDB movie reviews using the reward model you trained above.

Unlike supervised fine-tuning, RL allows model to generate it's own sentences on each training step. Then, it calculates the reward of those specific sentences, and finally, updates the model to increase the probability of sentences with high reward.

Thus, each RLHF consists of three stages: __Rollout__, __Evaluation__ and __Update__

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2_bert_training.png' width='600'>

The update stage depends on the specific RL algorithm. We'll be using Proximal Policy Optimization, or [PPO](https://arxiv.org/abs/1707.06347), similarly to what was used for InstructGPT.

Before we run those 3 stages, however, we need to create a dataset of "queries" - partial reviews in our case.

In [ ]:
import random
def sample_length(min_len=2, max_len=8):
    return random.randint(min_len, max_len)
# Note: this code is specific to IMDB; you will need to re-write it for other tasks
imdb_for_rlhf = imdb.filter(lambda row: len(row["text"]) > 200, batched=False)
imdb_for_rlhf = imdb_for_rlhf.remove_columns(["label"])

def select_query_and_tokenize(sample):
    q_len = sample_length(2, 8)   # ← call the function
    query_ids = main_tokenizer.encode(sample["text"])[:q_len]

    sample["query"] = main_tokenizer.decode(query_ids)
    sample["input_ids"] = query_ids
    return sample

imdb_for_rlhf = imdb_for_rlhf.map(select_query_and_tokenize, batched=False)
imdb_for_rlhf.set_format(type="torch")


Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/24895 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


Next, let's prepare your reward model to predict rewards on whatever reviews were generated. Note that we use plaintext reviews because main model uses a different tokenizer from the reward model.

In [ ]:
from typing import List
def compute_reward(texts: List[str]) -> torch.Tensor:
  inputs = reward_tokenizer(texts, truncation=True, padding=True, return_tensors='pt').to(device)
  with torch.no_grad():
    return reward_model(**inputs).logits[:, 0]

In [ ]:
compute_reward([imdb[45]['text'], imdb[16000]['text']])  # test on human-written reviews

tensor([16.6250, -2.1719], device='cuda:0')

Finally, we move to RL training. In this tutorial, we'll train LoRA adapters and not the full model.

In [ ]:
import peft
peft_config = peft.LoraConfig(
    task_type=peft.TaskType.CAUSAL_LM, r=32, lora_alpha=32, lora_dropout=0.0, inference_mode=False
)

# reload main model as AutoModelForCausalLMWithValueHead - with an extra head needed for PPO
main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

main_model = trl.AutoModelForCausalLMWithValueHead.from_pretrained("lvwerra/gpt2-imdb", device_map=device)
main_model = peft.get_peft_model(main_model, peft_config, adapter_name='default')
main_model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 125,620,225 || trainable%: 0.9391


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Same as before, trl has a special type of trainer that minimize PPO-specific pseudo-loss. You can read more on this trainer [here](https://huggingface.co/docs/trl/main/en/ppo_trainer).

In [ ]:
!nvidia-smi
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

Fri Feb  6 13:45:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             29W /   70W |    9216MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# del main_model
# del ref_model
# del value_model

device = "cuda" if torch.cuda.is_available() else "cpu"

main_tokenizer = transformers.AutoTokenizer.from_pretrained("lvwerra/gpt2-imdb")
main_tokenizer.pad_token = main_tokenizer.eos_token

from transformers import AutoModelForCausalLM
from trl import AutoModelForCausalLMWithValueHead

main_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    "lvwerra/gpt2-imdb"
).to(device)

ref_model = AutoModelForCausalLM.from_pretrained(
    "lvwerra/gpt2-imdb"
).to(device)

# value_model = AutoModelForCausalLM.from_pretrained(
#     "lvwerra/gpt2-imdb"
# ).to(device)

main__model.config.pad_token_id = main_tokenizer.eos_token_id
main__model.config.return_dict = True
ref_model.config.return_dict = True
# value_model.config.return_dict = True

# value_model.eval()
# for p in value_model.parameters(): p.requires_grad = False

ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False



NameError: name 'main__model' is not defined

In [ ]:
import torch
from types import SimpleNamespace
from transformers import GPT2LMHeadModel

class GPT2WithScore(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.config.return_dict = True

    @property
    def config(self):
        # Expose inner model's config
        return self.model.config

    def generate(self, **kwargs):
      return self.model.generate(**kwargs)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)

    def score(self, input_ids, attention_mask=None, **kwargs):
        # Compute logits
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        logits = outputs.logits

        # Fake value head (zeros for testing)
        value = torch.zeros(logits.shape[0], device=logits.device)

        # PPOTrainer expects an object with .logits and .value
        return SimpleNamespace(logits=logits, value=value)

policy_model = GPT2WithScore(i_model)

In [ ]:
from trl import AutoModelForCausalLMWithValueHead
import torch
from transformers import GenerationConfig

device = "cuda" if torch.cuda.is_available() else "cpu"


# main_model.base_model_prefix = "transformer"
policy_model.generation_config = GenerationConfig.from_pretrained("openai-community/gpt2")
main_tokenizer.eos_token = '<|endoftext|>'
print(main_tokenizer.encode('<|endoftext|>'))
main_tokenizer.eos_token_id = 50256


[50256]


In [ ]:
def collator(data):
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [d["input_ids"] for d in data],
            batch_first=True,
            padding_value=main_tokenizer.pad_token_id
        )
    }



training_args = trl.PPOConfig(
    # model_name=main_model.config._name_or_path,
    gradient_accumulation_steps=8,
    learning_rate=1.41e-5,
    batch_size=2,
    output_dir="results/"
    # ppo_epochs=4,                 # PPO performs this many updates per training batch
)

ppo_trainer = trl.PPOTrainer(
    training_args, model=policy_model, processing_class=main_tokenizer, ref_model=ref_model, reward_model=reward_model, value_model=value_model,
    train_dataset=imdb_for_rlhf, data_collator=collator
)
# note: we pass main_model.model because PPOTrainer checks for one of several supported model types ...
# ... main_model.model is a model with adapters, which is supported. main_model itself is a wrapper that is not supported

In [ ]:

ppo_trainer.train()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


===training policy===


AttributeError: 'GPT2LMHeadModel' object has no attribute 'score'

In [ ]:
## Tired of dependecy coflicts trl removed a step() and genearte(), and a new version fails to work with huggingadce new!!!
## I do not see any benefit from this library, so I will revrite PPO with RLHF in torch myself.
## it will be more beneficial

'''
--- roadmap ---

1) get model gpt imdb,
2) set LoRa (Q, V) and ml_head
3) train reward_model (already done earlier)
4) train on PPO (custom)

5) check results

--- how it words ---

! vale model - is just ml_head (used to distribute expected reward in token level)

-1-
get model genrate text:
main_model->
text: a b c d ...
logs: 0.8, 0.7 0.1 ...
values: x, x, x, ...

ref_model->
text: a b c d ...
logs: 0.6, 0.4 1.0 ...

-2-
get reward model scalar ex:(3.5)
rw = 3.5

-3-
KL penalty to not drift too far from og model!

KL = -B * (log(main_model(x))-log(ref_model(x)))
B - 0.1 or 0.05 common

KL(t) <- interm tokens
rw + KL(t) <- final token

-4-

G(t) = rt + y * r(t+1) + y^2 * r(t+2)+...

A(t) = G(t) - V(t)

if A>0 increase prob
else A<0 decrease prob

-5- Re-evaluate tokens:

log(π_new) = main_model(tokens)

-6- PPO ratio:

r_t = exp(logπ_new - logπ_old)

-7-

Loss(value)​=(V(t)−G(t))^2

-8-

ε = 0.1 or 0.2
Loss(policy​)=−min(r(t)*​A(t)​,clip(r(t)​,1−ϵ,1+ϵ)A(t)​)

-9-
c1 ≈ 0.5  (very common)
c1 ≈ 1.0  (also used)

L=L(policy)​+c1 * ​L(value)​−c2​ * entropy

'''

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
imdb = datasets.load_dataset("stanfordnlp/imdb", split='train')

tokenizer = main_tokenizer

def encode_batch(batch):
    return {"input_ids": [torch.tensor(tokenizer.encode(x)) for x in batch["text"]]}

encoded_dataset = imdb.map(encode_batch, batched=True, batch_size=32)

dataloader = DataLoader(
    encoded_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=lambda batch: {
        "input_ids": [item["input_ids"] for item in batch]
    }
)


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1168 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
# ----------------------------
# 2) Hyperparameters
# ----------------------------
beta = 0.05           # KL penalty
gamma = 1.0           # discount (usually 1.0)
epsilon = 0.2         # PPO clip
c1 = 0.5              # value loss weight
c2 = 0.01             # entropy weight


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PPOTrainer:
    def __init__(
        self,
        main_model,
        reference_model,
        value_model,
        reward_model,
        optimizer,
        beta=0.05,
        gamma=1.0,
        epsilon=0.2,
        c1=0.5,
        c2=0.01,
        device='cpu'
    ):
        self.main_model = main_model.to(device)
        self.reference_model = reference_model.to(device)
        self.value_model = value_model
        self.reward_model = reward_model.to(device)
        self.optimizer = optimizer
        self.beta = beta
        self.gamma = gamma
        self.epsilon = epsilon
        self.c1 = c1
        self.c2 = c2
        self.device = device

    @torch.no_grad()
    def compute_old_probs_and_kl(self, tokens, ref_log_probs):
        # main_model logits
        log_probs_old, values_old = self.main_model(tokens)
        logp_old = log_probs_old.gather(-1, tokens.unsqueeze(-1)).squeeze(-1)

        # KL penalty
        logp_ref = ref_log_probs.gather(-1, tokens.unsqueeze(-1)).squeeze(-1)
        kl_penalty = -self.beta * (logp_old - logp_ref)
        return logp_old, values_old, kl_penalty

    def compute_rewards(self, kl_penalty, batch_rewards):
        rewards = kl_penalty.clone()
        rewards[:, -1] += batch_rewards.to(self.device)
        return rewards

    def compute_returns_and_advantages(self, rewards, values_old):
        B, T = rewards.shape
        G = torch.zeros_like(rewards)
        A = torch.zeros_like(rewards)
        for b in range(B):
            G[b, -1] = rewards[b, -1]
            for t in reversed(range(T-1)):
                G[b, t] = rewards[b, t] + self.gamma * G[b, t+1]
            A[b] = G[b] - values_old[b]
        return G, A

    @torch.no_grad()
    def generate_responses(self, input_ids, max_new_tokens=64):
        self.main_model.eval()
        return self.main_model.gpt2.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            pad_token_id=self.tokenizer.eos_token_id
        )

    def train_step(self, tokens, ref_log_probs, batch_rewards):
      tokens = tokens.to(self.device)
      ref_log_probs = ref_log_probs.to(self.device)
      batch_rewards = batch_rewards.to(self.device)

      # --- 1. Prepare shifted inputs/targets ---
      input_tokens = tokens[:, :-1]
      target_tokens = tokens[:, 1:]
      T = target_tokens.size(1)

      # --- 2. Get old policy (detached) ---
      with torch.no_grad():
          logits_old, values_old = self.main_model(input_tokens)
          log_probs_old = F.log_softmax(logits_old, dim=-1)
          logp_old = log_probs_old.gather(-1, target_tokens.unsqueeze(-1)).squeeze(-1)

          logp_ref = ref_log_probs[:, :-1].gather(-1, target_tokens.unsqueeze(-1)).squeeze(-1)

          # KL penalty
          kl = logp_old - logp_ref
          rewards = -self.beta * kl  # Start with KL penalty at all timesteps
          rewards[:, -1] += batch_rewards  # Add scalar reward to final timestep

      G, A = self.compute_returns_and_advantages(rewards, values_old[:, :-1])
      A = (A - A.mean()) / (A.std() + 1e-8) # for stability
      A = A.clamp(-5, 5)

      logits_new, values_new = self.main_model(input_tokens)
      log_probs_new = F.log_softmax(logits_new, dim=-1)
      logp_new = log_probs_new.gather(-1, target_tokens.unsqueeze(-1)).squeeze(-1)

      probs_new = log_probs_new.exp()
      entropy = -(probs_new * log_probs_new).sum(-1).mean()

      ratio = torch.exp(logp_new - logp_old).clamp(0, 10) # stability clamp

      # Losses
      policy_loss = -torch.min(
          ratio * A,
          torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * A
      ).mean()
      value_loss = F.mse_loss(values_new[:, :-1], G)
      loss = policy_loss + self.c1 * value_loss - self.c2 * entropy

      # Optimize
      self.optimizer.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(self.main_model.parameters(), 1.0)
      self.optimizer.step()

      return loss.item(), policy_loss.item(), value_loss.item(), entropy.item()

In [ ]:
class GPT2WithValueHead(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.gpt2 = base_model
        self.value_head = nn.Linear(base_model.config.n_embd, 1)
        nn.init.zeros_(self.value_head.bias)

    def forward(self, input_ids):
        outputs = self.gpt2(
            input_ids,
            output_hidden_states=True,
            return_dict=True
        )
        # last hidden layer
        last_hidden = outputs.hidden_states[-1]  # [B, T, D]
        logits = outputs.logits                  # [B, T, V]
        values = self.value_head(last_hidden).squeeze(-1)
        return logits, values

    def generate(self, *args, **kwargs):
        return self.gpt2.generate(*args, **kwargs)



class GPT2Reference(nn.Module):
    def __init__(self, gpt2):
        super().__init__()
        self.gpt2 = gpt2
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask=None):
        outputs = self.gpt2(input_ids, attention_mask=attention_mask)
        return F.log_softmax(outputs.logits, dim=-1)


In [ ]:
from transformers import GPT2LMHeadModel

base_policy = GPT2LMHeadModel.from_pretrained("lvwerra/gpt2-imdb")
base_ref    = GPT2LMHeadModel.from_pretrained("lvwerra/gpt2-imdb")

policy_model = GPT2WithValueHead(base_policy)
reference_model = base_ref

In [ ]:
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)

trainer = PPOTrainer(
    main_model=policy_model,
    reference_model=reference_model,
    value_model=None,
    reward_model=reward_model,
    optimizer=optimizer,
    device="cuda",
    beta=0.05,
    gamma=1.0,
    epsilon=0.2,
    c1=0.05,
    c2=0.01
)

# B, T = 2, 8
# tokens = torch.randint(0, 50256, (B, T))
# ref_logits = torch.randn(B, T, 50256)
# ref_log_probs = F.log_softmax(ref_logits, dim=-1)
# batch_rewards = torch.tensor([3.5, 2.0])

# losses = trainer.train_step(tokens, ref_log_probs, batch_rewards)
# print("Losses:", losses)


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
def collate_fn(batch, max_prompt_len=511):
    texts = [x["text"] for x in batch]

    enc = main_tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_prompt_len,
        return_tensors="pt"
    )
    return enc["input_ids"], texts

@torch.no_grad()
def generate_responses(model, input_ids, max_new_tokens=64):
    model.eval()
    return model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        pad_token_id=main_tokenizer.eos_token_id
    )

@torch.no_grad()
def get_ref_log_probs(ref_model, tokens):
    outputs = ref_model(tokens)
    logits = outputs.logits              # [B, T, V]
    log_probs = F.log_softmax(logits, dim=-1)
    return log_probs


@torch.no_grad()
def compute_rewards_bert(reward_model, sequences):
    texts = main_tokenizer.batch_decode(sequences, skip_special_tokens=True)

    enc = reward_tokenizer(
        texts,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(next(reward_model.parameters()).device)

    outputs = reward_model(**enc)

    # binary classifier case
    if outputs.logits.shape[-1] == 2:
        rewards = torch.softmax(outputs.logits, dim=-1)[:, 1]
    else:
        rewards = outputs.logits.squeeze(-1)

    return rewards


In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

imdb_test = load_dataset("stanfordnlp/imdb", split="test") # smaller I already almost reached OMM and trains paifully slow, so I will train only on test with batch 4

loader = DataLoader(
    imdb,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)


In [ ]:
main_tokenizer.pad_token = main_tokenizer.eos_token
main_tokenizer.padding_side = "left"

In [ ]:
!export CUDA_LAUNCH_BLOCKING=1
import torch
torch.cuda.empty_cache()      # frees cached memory
torch.cuda.reset_peak_memory_stats()  # optional, resets peak stats
import gc
gc.collect()
torch.cuda.empty_cache()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True



In [ ]:
for step, (prompt_ids, _) in enumerate(loader):
    prompt_ids = prompt_ids.to(trainer.device)

    # 1) generate sequences
    full_sequences = generate_responses(policy_model, prompt_ids, max_new_tokens=64)

    # 2) compute reference log probs
    ref_log_probs = get_ref_log_probs(trainer.reference_model, full_sequences)

    # 3) compute rewards
    rewards = compute_rewards_bert(trainer.reward_model, full_sequences)
    rewards = rewards.clamp(-1, 1)  # clip again for safety

    # 4) PPO train step
    loss_total, policy_loss, value_loss, entropy = trainer.train_step(
        tokens=full_sequences,
        ref_log_probs=ref_log_probs,
        batch_rewards=rewards
    )

    # 5) debug info every few steps
    if step % 5 == 0:
        print(f"step {step}/{len(loader)} | loss {loss_total:.4f}")
        print(f"policy_loss {policy_loss:.4f} | value_loss {value_loss:.4f} | entropy {entropy:.4f} | avg_reward {rewards.mean():.4f}\n")

step 0/6250 | loss 24462911355653646548956449079296.0000
policy_loss -0.0000 | value_loss 41711.8906 | entropy -2446291299979276122485212667969536.0000 | avg_reward 1.0000



AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Negative loss = policy is improving on reward (so model learn to make negative rewiews)
# Positive loss = policy would be penalized / pulled back (if model makes too much positive rewiews)

# (model is improving on the task) if reward also improves (max was 5.xxx I think for reward model to not start to exploit it)
# loss not getting smaller is normal, it should stabilize and that is considered convergence in RLHF
# will train till 200 (should be enough + not overtrain to too much toxicity!)

In [ ]:
def sample_order(model,reward_model,tok,tok_rew,N=8,start_with="It was",device="cuda"):
    model.eval()
    reward_model.eval()

    inputs = tok([start_with] * N,return_tensors="pt").to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True
        )

    sentences = tok.batch_decode(generated,skip_special_tokens=True)

    reward_inputs = tok_rew(sentences,padding=True,truncation=True,max_length=512,return_tensors="pt").to(device)

    with torch.no_grad():
        logits = reward_model(**reward_inputs).logits
        scores = logits[:, 0]

    ranked = sorted(zip(sentences, scores.tolist()),key=lambda x: x[1],reverse=True)

    for text, score in ranked:
        print(f"{score:.4f} | {text}")


sample_order(policy_model, reward_model, main_tokenizer, reward_tokenizer)

## Main assignment - <u>actually</u> train the model (8 points)


Your main task for this week is to use the RLHF pipeline to train a model for a reward of your choice. Here's what you can choose from:

__A. Toxicity fine-tuning:__ train the model to be less (or more!) toxic. For this task, you may use the data from [jigsaw toxic comments](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge) and [lmsys/toxic-chat](https://huggingface.co/datasets/lmsys/toxic-chat),  or any other source. Alternatively, you may use toxicity scores from [oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1).


__B. Actual human feedback:__ use one of the existing datasets with pairwise human feedback to align your langauge model. You may use [anthropic's hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf), [OpenAssistant dataset](https://huggingface.co/datasets/OpenAssistant/oasst1) or any other data you see fit. You may also turn the tables and train the model to [minimize](https://habrastorage.org/getpro/geektimes/post_images/ac7/2ad/827/ac72ad82767d4132164a4b6b76196c42.jpg) human preferences, as long as your model does not degrade to gibberish.

__C. Controlled generation:__ Instead of training a reward model from human feedback, you may define the reward function as the text length (longer or shorter) or number of times the model uses specific words (e.g. "sorry", "apologize"). If you choose specific words, make sure the model generates them at least sometimes.

__Alternatively,__ you may choose a different task. However, unless your task is very similar to one of the above, there is a chance that it will be **significantly** harder to solve, requiring orders of magnitude more compute and tuning. If you are in doubt, please ask the course staff. If they are AFK (again >.<), please prefer one of the recommended tasks.


#### General tips & tricks


Things to look out for:
- during PPO stage, the reward model should be in eval mode (dropout disabled)
- make sure max_length and max_new_tokens are enough for your chosen dataset - at least most of the time
- when in doubt, view the data manually or inspect how the model performs on a few samples


We highly recommend that you manually check the performance after each sub-stage:
1. when you assembled the pairwise dataset, inspect a couple of from of *your* dataset class and detokenize them. Make sure that you-the-human understand why one sample was accepted and the other - rejected. At least most of the time. This also lets you spot tokenization/truncation errors.
2. after you trained a reward model, measure how accurate this model is in isolation. If your reward model is poor, any subsequent RLHF will also fail.
3. once you've trained the main model with RL, ask it to generate examples and explore how well it does. If it produces an obviously bad output, check if the reward model assigns high reward to that output. If yes, reward model is the culprit; if no, it's a question of better/longer PPO training.

__It is also a good idea to periodically print samples during training.__

__When stuck, simplify the problem.__ If you've spent a several hours enchanting the reward model but it still won't budge, try switching to a simple subtask. For instance, if you're training on hh-rlhf, try limiting it the dataset to 10% of the shortest sequences - they are typically easier to learn.


## Assignment stages (and grading)

Regardless of the specific task you chose, your solution needs to contain several parts that will be graded separately.


#### Stage 1: reward model (4 points)

Construct a dataset for training the reward model on your problem. Then, train a reward model on that dataset and evaluate how well can your model predict preferences on a hold-out (test) subset of your data.

Please make sure that the part of your notebook where you evaluate reward model is clearly visible and reasonably easy to read. And for all that is holy, do not call it IMDB unless it actually **is** data of imdb movie reviews :)

__Not all tasks require a reward model for later PPO fine-tuning.__ For instance, there's no reason to train a reward model if your reward equals sentence length. Likewise, toxicity reward can be estimated with a pre-trained toxicity classifier. __If your task does not require training a reward model, please train an unrelated model on [hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) as though you were solving assignment version B.__ This is for grading purposes only, you won't use this model for stage 2.


#### Stage 2: RL fine-tuning (4 points)

Once the reward model is ready - or you can compute rewards without a model - it is time to maximize that reward with PPO. Optionally, you may replace PPO with another RL algorithm (or unlikelihood learning scheme), but only if you're feeling adventurous.


First, you need to choose a language model to be fine-tuned. You may choose any model, but make sure that your model **can** generate the data in your format. For instance, [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) is a general purpose LM and may (or may not) need prompt engineering to generate chat assistant responses. For that reason, it is best if you **do not use `"lvwerra/gpt2-imdb"` unless you're generating only movie reviews**.



There are two "difficulty modes" for this task:
For the **easy mode**, use [gpt2-large](https://huggingface.co/gpt2-large) or [opt-1.3b](https://huggingface.co/facebook/opt-1.3b) with minimal code changes.
If you want the **Hard mode:** use a larger (e.g. 7B) model in combination with `load_in_4bit` and LoRA, the same way we did last week.
Some reasonable model choices are [LLaMA-7B](https://huggingface.co/Enoch/llama-7b-hf), [Falcon-7b](https://huggingface.co/tiiuae/falcon-7b), [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) for general-purpose LM or [guanaco-7b](https://huggingface.co/timdettmers/guanaco-7b), [vicuna-7b](https://huggingface.co/lmsys/vicuna-7b-v1.5) for chat-based tasks, though there are many more (see [leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)). In the hard mode, you will need to modify the training arguments to enable 4-bit fine-tuning. Furthermore, your experiments will take somewhat longer to complete. On the plus side, your model will produce significantly better results.

__High reward is not enough!__ RL algorithms are famous for [cheating their reward functions](https://openai.com/research/faulty-reward-functions). To ensure that your model is actually doing what you want it to do, you will need some additional evaluation. To get the full grade, provide at least 20 side-by-side examples of your fine-tuned model vs original model predictions and a short summary.

Alternatively, you may provide 5 examples and some extrinsic evaluation metric over many examples. For instance, you may use a different pre-trained toxicity score for option A. When dealing with human preferences, you may choose to [enlist actual humans](https://toloka.ai/) or [ask GPT4/Claude](https://arxiv.org/pdf/2304.03277.pdf) to compare your model's predictions. For task C, when optimizing for simple rewards like sentence lengths, it is enough to compare histograms of rewards (e.g. average lengths).










